# Build Gym tasks with Data Designer

This notebook rebuilds the NeMo Gym Workplace Assistant data-generation example as a normal Data Designer workflow. `GymTaskProcessorConfig` maps generated columns into Gym's task envelope without a custom Python column.

Set `GYM_ROOT` to a Gym checkout and `NVIDIA_API_KEY` to a Build API key before running it.

In [ ]:
import json
import os
from pathlib import Path

import data_designer.config as dd
import pandas as pd
from data_designer.interface import DataDesigner
from pydantic import BaseModel, Field, field_validator

from data_designer_gym.config import GymTaskProcessorConfig

GYM_ROOT = Path(os.environ["GYM_ROOT"]).expanduser().resolve()
TOOLS_DIR = GYM_ROOT / "resources_servers/workplace_assistant/notebooks/synthetic-data-generation/tools"
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "artifacts/workplace-assistant"))
NUM_RECORDS = int(os.environ.get("NUM_RECORDS", "10"))

if not TOOLS_DIR.is_dir():
    raise ValueError(f"Workplace Assistant assets not found: {TOOLS_DIR}")
if not os.environ.get("NVIDIA_API_KEY"):
    raise ValueError("NVIDIA_API_KEY is required")

## 1. Build environment seeds

Gym already owns the environment definition. We turn its tool schemas and common task patterns into ordinary Data Designer seed rows.

In [ ]:
TOOL_FILES = [
    "company_directory.json",
    "email.json",
    "calendar.json",
    "analytics.json",
    "project_management.json",
    "customer_relationship_manager.json",
]

environment = json.loads((TOOLS_DIR / "environment.json").read_text())
tool_groups = [json.loads((TOOLS_DIR / name).read_text()) for name in TOOL_FILES]
all_tools = [tool for group in tool_groups for tool in group["tools"]]
tool_categories = {tool["name"]: group["database"] for group in tool_groups for tool in group["tools"]}
company_tools = {tool["name"] for tool in tool_groups[0]["tools"]}
patterns = environment["common_multi_step_patterns"]

seed_rows = []
for seed_id in range(NUM_RECORDS):
    pattern = patterns[seed_id % len(patterns)]
    selected_names = set(pattern.get("tools_used", [])) | company_tools
    selected_tools = [tool for tool in all_tools if tool["name"] in selected_names]
    category = next((tool_categories[name] for name in pattern.get("tools_used", [])), "general")
    seed_rows.append(
        {
            "seed_id": seed_id,
            "category": category,
            "pattern": pattern["pattern"],
            "pattern_description": pattern["description"],
            "tools_json": json.dumps(selected_tools),
            "system_prompt": environment["system_prompt"],
        }
    )

seeds = pd.DataFrame(seed_rows)
seeds.head(2)

## 2. Define the generated fields

The workflow generates a user request and a compact reference plan. The plan doubles as an early feasibility check; Gym still owns runtime verification.

In [ ]:
class ToolCall(BaseModel):
    name: str
    arguments: str

    @field_validator("arguments")
    @classmethod
    def validate_arguments(cls, value: str) -> str:
        if not isinstance(json.loads(value), dict):
            raise ValueError("arguments must encode a JSON object")
        return value


class ReferencePlan(BaseModel):
    tool_calls: list[ToolCall] = Field(min_length=1, max_length=6)
    is_valid: bool
    issues: str


USER_QUERY_PROMPT = """
Create one natural workplace request for this task pattern.

Pattern: {{ pattern }}
Pattern description: {{ pattern_description }}
Available tools: {{ tools_json }}

The request must be achievable with the tools, use schema-valid values, require no more than six tool calls,
and not mention tool names. Return only the request.
"""

REFERENCE_PLAN_PROMPT = """
Create the minimal executable tool-call plan for this request.

System context: {{ system_prompt }}
Request: {{ user_query }}
Available tools: {{ tools_json }}

Use exact tool names and encode each schema-valid argument object as a JSON string. Set is_valid to false
when the request cannot be completed with these tools. Return only ReferencePlan.
"""

## 3. Map generated columns into Gym tasks

The processor assembles the common Gym task structure declaratively. Environment-specific fields remain ordinary column mappings or fixed values.

In [ ]:
builder = dd.DataDesignerConfigBuilder(
    model_configs=[
        dd.ModelConfig(
            alias="generator",
            model="openai/gpt-oss-20b",
            provider="nvidia-build",
            inference_parameters=dd.ChatCompletionInferenceParams(
                max_tokens=4096,
                temperature=0.6,
                max_parallel_requests=1,
            ),
        )
    ]
)
builder.with_seed_dataset(dd.DataFrameSeedSource(df=seeds), sampling_strategy=dd.SamplingStrategy.ORDERED)
builder.add_column(dd.LLMTextColumnConfig(name="user_query", prompt=USER_QUERY_PROMPT, model_alias="generator"))
builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="reference_plan",
        prompt=REFERENCE_PLAN_PROMPT,
        output_format=ReferencePlan,
        model_alias="generator",
    )
)
builder.add_column(dd.ExpressionColumnConfig(name="gym_category", expr="workplace_assistant_{{ category }}"))
builder.add_processor(
    GymTaskProcessorConfig(
        name="gym_tasks",
        messages=[
            {"role": "system", "content_column": "system_prompt"},
            {"role": "user", "content_column": "user_query"},
        ],
        tools_column="tools_json",
        tool_fields=["type", "name", "description", "parameters", "strict"],
        response_params={"parallel_tool_calls": False, "temperature": 1.0},
        task_columns={
            "ground_truth": "reference_plan.tool_calls",
            "category": "gym_category",
            "pattern": "pattern",
            "qualification": "reference_plan",
        },
        task_values={"environment_name": "workplace_assistant"},
        include="reference_plan.is_valid",
        provenance_columns=["seed_id", "category", "pattern"],
        scenario_namespace="workplace-assistant",
    )
)

## 4. Generate and export

The processor runs inside the Data Designer workflow. Its `task_json` artifact is already one complete Gym task, so exporting JSONL is a direct write.

In [ ]:
provider = dd.ModelProvider(
    name="nvidia-build",
    endpoint="https://integrate.api.nvidia.com/v1",
    provider_type="openai",
    api_key="NVIDIA_API_KEY",
)
designer = DataDesigner(artifact_path=OUTPUT_DIR / "data-designer", model_providers=[provider])
results = designer.create(
    builder,
    num_records=NUM_RECORDS,
    dataset_name="workplace-assistant-gym-tasks",
)

artifacts = results.load_processor_dataset("gym_tasks")
tasks_path = OUTPUT_DIR / "gym-tasks.jsonl"
tasks_path.parent.mkdir(parents=True, exist_ok=True)
tasks_path.write_text("\n".join(artifacts["task_json"].tolist()) + "\n")

print(f"Exported {len(artifacts)} tasks to {tasks_path.resolve()}")
json.loads(artifacts.iloc[0]["task_json"])

## Run with Gym

Pass the generated `gym-tasks.jsonl` to Gym's native rollout command. After collection, `data-designer-gym ingest` can join rollouts and failure sidecars back to the stable Data Designer scenario IDs.